# COMP8240 — Step 1: PatchCore replication on MVTec AD (all 15 categories)

**Before running:** Runtime → Change runtime type → **T4 GPU**.

This notebook:
1. records the exact environment (needed for the reproducibility discussion),
2. runs PatchCore on every MVTec AD category and saves results to Google Drive after **each** category (so a Colab disconnect doesn't lose work — just re-run and it resumes),
3. compares your numbers with the paper,
4. saves charts and heatmaps for the slides,
5. (optional) runs a small coreset-ratio ablation.

In [ ]:
!pip install -q anomalib
from google.colab import drive
drive.mount('/content/drive')

## 1. Record the environment
This writes `environment.json` and a full `requirements_frozen.txt` to Drive. You will quote these in the presentation and report.

In [ ]:
import os, sys, json, platform, datetime, torch, anomalib, lightning

OUT_DIR = "/content/drive/MyDrive/COMP8240_PatchCore/step1_mvtec"
os.makedirs(OUT_DIR, exist_ok=True)

env = {
    "date": datetime.datetime.now().isoformat(timespec="minutes"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
    "anomalib": anomalib.__version__,
    "lightning": lightning.__version__,
}
print(json.dumps(env, indent=2))
json.dump(env, open(f"{OUT_DIR}/environment.json", "w"), indent=2)
!pip freeze > "{OUT_DIR}/requirements_frozen.txt"

## 2. Configuration
Anomalib's PatchCore defaults match the paper's main setup: WideResNet-50 backbone, features from layer2 + layer3, images resized to 256 and centre-cropped to 224. We use a **10% coreset**, so compare against the paper's *PatchCore-10%* row.

In [ ]:
CATEGORIES = ["bottle", "cable", "capsule", "carpet", "grid", "hazelnut",
              "leather", "metal_nut", "pill", "screw", "tile", "toothbrush",
              "transistor", "wood", "zipper"]
CORESET_RATIO = 0.1
SEED = 42
DATA_ROOT = "/content/datasets/MVTecAD"   # local disk: much faster than Drive
RESULTS_CSV = f"{OUT_DIR}/mvtec_results.csv"

## 3. Run function
Computes image AUROC, image F1, pixel AUROC and PRO (AUPRO, integrated to 30% FPR as in the paper). If PRO fails on your Anomalib version, it falls back to the default metrics and tells you.

In [ ]:
import gc, time
import pandas as pd
from lightning.pytorch import seed_everything
from anomalib.data import MVTecAD
from anomalib.models import Patchcore
from anomalib.engine import Engine

def build_model(ratio, with_pro):
    if not with_pro:
        return Patchcore(coreset_sampling_ratio=ratio)
    from anomalib.metrics import AUROC, AUPRO, F1Score, Evaluator
    metrics = [
        AUROC(fields=["pred_score", "gt_label"], prefix="image_"),
        F1Score(fields=["pred_label", "gt_label"], prefix="image_"),
        AUROC(fields=["anomaly_map", "gt_mask"], prefix="pixel_"),
        AUPRO(fields=["anomaly_map", "gt_mask"], prefix="pixel_"),
    ]
    return Patchcore(coreset_sampling_ratio=ratio,
                     evaluator=Evaluator(test_metrics=metrics))

def _run(cat, ratio, with_pro, tag):
    seed_everything(SEED, workers=True)
    dm = MVTecAD(root=DATA_ROOT, category=cat, num_workers=2)
    model = build_model(ratio, with_pro)
    engine = Engine(accelerator="gpu", devices=1, enable_progress_bar=False,
                    default_root_dir=f"/content/results/{tag}")
    t0 = time.perf_counter(); engine.fit(model=model, datamodule=dm)
    fit_s = time.perf_counter() - t0
    t0 = time.perf_counter(); res = engine.test(model=model, datamodule=dm)[0]
    test_s = time.perf_counter() - t0
    try: n_test = len(dm.test_data)
    except Exception: n_test = None
    try: bank = int(model.model.memory_bank.shape[0])
    except Exception: bank = None
    row = {"tag": tag, "category": cat, "coreset_ratio": ratio,
           **{k: round(float(v), 4) for k, v in res.items()},
           "fit_s": round(fit_s, 1), "test_s": round(test_s, 1),
           "n_test_images": n_test,
           "s_per_test_image": round(test_s / n_test, 3) if n_test else None,
           "memory_bank_size": bank}
    del model, engine, dm; gc.collect(); torch.cuda.empty_cache()
    return row

def run_category(cat, ratio=CORESET_RATIO, with_pro=True, tag="main"):
    try:
        return _run(cat, ratio, with_pro, tag)
    except Exception as e:
        if not with_pro: raise
        print(f"  PRO metric failed ({e!r}); re-running {cat} without PRO")
        gc.collect(); torch.cuda.empty_cache()
        return _run(cat, ratio, False, tag)

def append_row(row, path):
    df = pd.DataFrame([row])
    df.to_csv(path, mode="a", header=not os.path.exists(path), index=False)

## 4. Run all categories (resumable)
First run downloads MVTec AD (~5 GB, roughly 5–10 min). Each category then takes a few minutes. **If Colab disconnects, re-run cells 1–4 and it skips finished categories.**

In [ ]:
done = set()
if os.path.exists(RESULTS_CSV):
    done = set(pd.read_csv(RESULTS_CSV).query("tag == 'main'")["category"])

for cat in CATEGORIES:
    if cat in done:
        print(f"skip {cat} (already done)"); continue
    print(f"=== {cat} ===")
    row = run_category(cat)
    print({k: row[k] for k in row if k.startswith(("image_", "pixel_"))})
    append_row(row, RESULTS_CSV)

## 5. Compare with the paper
Fill `PAPER` with the **PatchCore-10%** per-category values from the paper (image AUROC in Table S1 / supplementary per-class table, pixel AUROC likewise), as percentages. Check the averages below against Table 1 of the paper too. Leave `None` for anything you haven't filled — the code handles it.

In [ ]:
# (image AUROC %, pixel AUROC %) for PatchCore-10% — copy from the paper
PAPER = {c: (None, None) for c in CATEGORIES}
# e.g. PAPER["bottle"] = (100.0, 98.6)

PAPER_MEAN = {"image_AUROC": 99.0, "pixel_AUROC": 98.1}  # verify against paper Table 1

df = pd.read_csv(RESULTS_CSV).query("tag == 'main'").set_index("category").loc[CATEGORIES]
cols = [c for c in ["image_AUROC", "image_F1Score", "pixel_AUROC", "pixel_AUPRO"] if c in df]
table = (df[cols] * 100).round(1)
table["paper_image_AUROC"] = [PAPER[c][0] for c in table.index]
table["paper_pixel_AUROC"] = [PAPER[c][1] for c in table.index]
table["diff_image"] = (table["image_AUROC"] - table["paper_image_AUROC"]).round(1)
table["diff_pixel"] = (table["pixel_AUROC"] - table["paper_pixel_AUROC"]).round(1)
table.loc["MEAN"] = table.mean(numeric_only=True).round(1)
table.to_csv(f"{OUT_DIR}/comparison_table.csv")
print("Paper means (10%):", PAPER_MEAN)
display(table)
display(df[["fit_s", "test_s", "s_per_test_image", "memory_bank_size"]])

## 6. Chart for the slides

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.arange(len(CATEGORIES)); w = 0.4
ours = (df["image_AUROC"] * 100).values
paper = [PAPER[c][0] if PAPER[c][0] is not None else np.nan for c in CATEGORIES]

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(x - w/2, ours, w, label="Our replication")
ax.bar(x + w/2, paper, w, label="Paper (PatchCore-10%)")
ax.set_xticks(x); ax.set_xticklabels(CATEGORIES, rotation=45, ha="right")
ax.set_ylabel("Image AUROC (%)"); ax.set_ylim(80, 101); ax.legend(loc="lower left")
ax.set_title("PatchCore on MVTec AD: replication vs paper")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/auroc_vs_paper.png", dpi=200); plt.show()

## 7. Collect heatmaps for the slides
Copies 2 defective + 1 good prediction image per category into Drive.

In [ ]:
import glob, shutil
HEAT_DIR = f"{OUT_DIR}/heatmaps"; os.makedirs(HEAT_DIR, exist_ok=True)
for cat in CATEGORIES:
    imgs = sorted(glob.glob(f"/content/results/main/**/{cat}/**/*.png", recursive=True))
    bad  = [p for p in imgs if "/good/" not in p][:2]
    good = [p for p in imgs if "/good/" in p][:1]
    for p in bad + good:
        kind = "good" if "/good/" in p else p.split("/")[-2]
        shutil.copy(p, f"{HEAT_DIR}/{cat}_{kind}_{os.path.basename(p)}")
print(len(os.listdir(HEAT_DIR)), "images saved to", HEAT_DIR)

## 8. (Optional) Coreset ablation
Reproduces the paper's claim that a much smaller memory bank barely hurts accuracy. Three categories × three ratios, PRO skipped to save time.

In [ ]:
ABL_CATS = ["bottle", "screw", "carpet"]
ABL_RATIOS = [0.01, 0.1, 0.25]
ABL_CSV = f"{OUT_DIR}/ablation_results.csv"

done = set()
if os.path.exists(ABL_CSV):
    d = pd.read_csv(ABL_CSV); done = set(zip(d.category, d.coreset_ratio))
for cat in ABL_CATS:
    for r in ABL_RATIOS:
        if (cat, r) in done: continue
        print(f"=== {cat} @ {r} ===")
        append_row(run_category(cat, ratio=r, with_pro=False, tag="ablation"), ABL_CSV)

abl = pd.read_csv(ABL_CSV)
display(abl.pivot_table(index="category", columns="coreset_ratio",
        values=["image_AUROC", "pixel_AUROC", "memory_bank_size", "s_per_test_image"]).round(3))